# ЛР № 3. Качество данных Airbnb Amsterdam
Притужалов Илья, Аглушевич Павел, Букатка Илона · ПИН-241.

**Вопрос:** как различаются заявленные суточные цены и календарная доступность
между районами при сравнении одинаковых типов жилья?
Одна строка — объявление, ключ — `id` внутри снимка.
[Project Brief](../docs/project_brief.md) · [Отчёт](../docs/data_quality_report.md).
Обрабатывается только участок raw → staging.

Для сдачи выберите ядро `.venv/bin/python`, выполните **Restart Kernel → Run All**
дважды с одними параметрами и сохраните выводы. Локальная диагностика в
`docs/evidence/profile_quality_local.ipynb` не заменяет запуск через S3.

In [1]:
RAW_PATH = "s3://raw/airbnb_amsterdam_listings/ingested_on=2026-10-23/listings.csv.gz"
ANALYTICAL_PERIOD = ("2026-06-15", "2026-06-24")
OUTPUT_PREFIX = "s3://staging/airbnb_amsterdam_listings/source_ingested_on=2026-10-23/"

## 1. Подключение и вход
Настройки читаются из `.env`, ключи не выводятся. Подключение общее со скриптом ЛР № 2.
CSV читается напрямую из S3. Исходные поля в памяти сохраняются текстом, чтобы
карантин сохранял точное представление значений. Фактические типы отдельно определяются
DuckDB по исходному CSV до нормализации. `hive_partitioning=false` исключает
добавление столбцов из пути. S3-объект raw не изменяется.

In [2]:
import hashlib
import json
import os
import sys
from datetime import date
from pathlib import Path
from urllib.parse import urlsplit

import boto3
import duckdb
from dotenv import load_dotenv

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").exists() and (p / "src").is_dir()
)
load_dotenv(ROOT / ".env", override=False)
sys.path.insert(0, str(ROOT / "src"))
from common.s3 import configure_s3  # noqa: E402


def show_table(columns, rows):
    # Без зависимости от pandas; в Jupyter показывается Markdown-таблица.
    def escape(value):
        return str(value).replace("|", "/").replace("\n", " ")

    text = "| " + " | ".join(columns) + " |\n"
    text += "| " + " | ".join("---" for _ in columns) + " |\n"
    text += "\n".join("| " + " | ".join(map(escape, row)) + " |" for row in rows)
    try:
        from IPython.display import Markdown, display

        display(Markdown(text))
    except ImportError:
        print(text)


def show_sql(query, params=None):
    result = con.execute(query, params or [])
    columns = [column[0] for column in result.description]
    rows = result.fetchall()
    show_table(columns, rows)
    return rows


def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"


raw_url = urlsplit(RAW_PATH)
assert raw_url.scheme == "s3" and raw_url.netloc == "raw"
source_date = next(
    p.split("=", 1)[1] for p in raw_url.path.split("/") if p.startswith("ingested_on=")
)
date.fromisoformat(source_date)
assert OUTPUT_PREFIX == (
    f"s3://staging/airbnb_amsterdam_listings/source_ingested_on={source_date}/"
)
assert date.fromisoformat(ANALYTICAL_PERIOD[0]) <= date.fromisoformat(ANALYTICAL_PERIOD[1])
endpoint = os.environ["S3_ENDPOINT"]
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint if "://" in endpoint else "http://" + endpoint,
    region_name="us-east-1",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
)


def raw_fingerprint():
    response = s3.get_object(Bucket="raw", Key=raw_url.path.lstrip("/"))
    with response["Body"] as body:
        digest = hashlib.sha256(body.read()).hexdigest()
    return {"bytes": response["ContentLength"], "sha256": digest}


raw_before = raw_fingerprint()
con = duckdb.connect()
configure_s3(con)
con.execute("SET preserve_insertion_order = true")
EXECUTION_ENVIRONMENT = "project S3"
print("Среда:", EXECUTION_ENVIRONMENT)
print("Вход:", RAW_PATH)
print("Выход:", OUTPUT_PREFIX)

Среда: project S3
Вход: s3://raw/airbnb_amsterdam_listings/ingested_on=2026-10-23/listings.csv.gz
Выход: s3://staging/airbnb_amsterdam_listings/source_ingested_on=2026-10-23/


In [3]:
con.execute(
    """
    CREATE TABLE raw AS
    SELECT * FROM read_csv_auto(?, all_varchar=true,
                                hive_partitioning=false, sample_size=-1)
""",
    [RAW_PATH],
)
inferred_schema = con.execute(
    """
    DESCRIBE SELECT * FROM read_csv_auto(?, hive_partitioning=false, sample_size=-1)
""",
    [RAW_PATH],
).fetchall()
raw_columns = [row[0] for row in con.execute("DESCRIBE raw").fetchall()]
raw_count = con.execute("SELECT count(*) FROM raw").fetchone()[0]
show_sql("""
    SELECT id, neighbourhood_cleansed, room_type, price,
           availability_365, has_availability FROM raw
    ORDER BY try_cast(id AS BIGINT) LIMIT 5
""")

| id | neighbourhood_cleansed | room_type | price | availability_365 | has_availability |
| --- | --- | --- | --- | --- | --- |
| 28871 | Centrum-West | Private room | $94.00 | 11 | t |
| 29051 | Centrum-Oost | Private room | None | 2 | t |
| 44129 | Centrum-West | Entire home/apt | $313.67 | 3 | t |
| 44391 | Centrum-Oost | Entire home/apt | None | 0 | t |
| 48373 | Buitenveldert - Zuidas | Entire home/apt | None | 0 | t |

[('28871', 'Centrum-West', 'Private room', '$94.00', '11', 't'),
 ('29051', 'Centrum-Oost', 'Private room', None, '2', 't'),
 ('44129', 'Centrum-West', 'Entire home/apt', '$313.67', '3', 't'),
 ('44391', 'Centrum-Oost', 'Entire home/apt', None, '0', 't'),
 ('48373', 'Buitenveldert - Zuidas', 'Entire home/apt', None, '0', 't')]

## 2. Профиль до преобразований
Пять основных полей: район, тип жилья, цена, число доступных дней, признак календаря.
Ключ и даты сбора рассматриваются дополнительно. Пропуск — NULL/пустая строка после trim.
Для каждого выбранного поля знаменатель доли пропусков — все строки.

In [4]:
selected_fields = [
    "neighbourhood_cleansed",
    "room_type",
    "price",
    "availability_365",
    "has_availability",
]
show_table(["Строк", "Исходных полей"], [(raw_count, len(raw_columns))])
schema_by_name = {row[0]: row[1] for row in inferred_schema}
profile_rows = []
for field in selected_fields:
    checked, missing, share = con.execute(f"""
        SELECT count(*), count(*) FILTER (WHERE nullif(trim({field}), '') IS NULL),
               count(*) FILTER (WHERE nullif(trim({field}), '') IS NULL)::DOUBLE
                 / nullif(count(*), 0)
        FROM raw
    """).fetchone()
    examples = con.execute(f"""
        SELECT DISTINCT {field} FROM raw
        WHERE nullif(trim({field}), '') IS NOT NULL ORDER BY {field} LIMIT 2
    """).fetchall()
    profile_rows.append(
        [
            field,
            schema_by_name[field],
            "; ".join(row[0] for row in examples),
            checked,
            missing,
            share,
        ]
    )
show_table(["Поле", "Тип DuckDB", "Примеры", "Проверено", "Пропусков", "Доля"], profile_rows)

| Строк | Исходных полей |
| --- | --- |
| 10369 | 90 |

| Поле | Тип DuckDB | Примеры | Проверено | Пропусков | Доля |
| --- | --- | --- | --- | --- | --- |
| neighbourhood_cleansed | VARCHAR | Bijlmer-Centrum; Bijlmer-Oost | 10369 | 0 | 0.0 |
| room_type | VARCHAR | Entire home/apt; Hotel room | 10369 | 0 | 0.0 |
| price | VARCHAR | $1,008.33; $1,013.52 | 10369 | 3992 | 0.3849937313144951 |
| availability_365 | BIGINT | 0; 1 | 10369 | 0 | 0.0 |
| has_availability | BOOLEAN | f; t | 10369 | 136 | 0.01311601890249783 |

In [5]:
show_sql("""
    WITH keys AS (
        SELECT nullif(trim(id), '') AS business_key, count(*) AS n FROM raw GROUP BY 1
    )
    SELECT coalesce(sum(n) FILTER (WHERE business_key IS NULL), 0) AS missing_ids,
           count(*) FILTER (WHERE business_key IS NOT NULL AND n > 1) AS duplicate_keys,
           coalesce(sum(n-1) FILTER (WHERE business_key IS NOT NULL AND n > 1), 0)
             AS excess_duplicate_rows
    FROM keys
""")
show_sql("""
    SELECT min(try_cast(last_scraped AS DATE)) AS first_listing_scrape,
           max(try_cast(last_scraped AS DATE)) AS last_listing_scrape,
           min(try_cast(calendar_last_scraped AS DATE)) AS first_calendar_scrape,
           max(try_cast(calendar_last_scraped AS DATE)) AS last_calendar_scrape FROM raw
""")
# Период описывает полный снимок: строки вне него не удаляются молча.
show_sql(
    """
    SELECT count(*) FILTER (WHERE try_cast(last_scraped AS DATE) IS NULL
        OR try_cast(last_scraped AS DATE) NOT BETWEEN ?::DATE AND ?::DATE)
          AS listing_dates_outside_period,
        count(*) FILTER (WHERE try_cast(calendar_last_scraped AS DATE) IS NULL
        OR try_cast(calendar_last_scraped AS DATE) NOT BETWEEN ?::DATE AND ?::DATE)
          AS calendar_dates_outside_period FROM raw
""",
    [*ANALYTICAL_PERIOD, *ANALYTICAL_PERIOD],
)
show_sql("""
    SELECT last_scraped, count(*) AS listings FROM raw
    GROUP BY last_scraped ORDER BY last_scraped
""")
show_sql("""
    SELECT room_type, count(*) AS listings,
           round(100.0 * count(*) / (SELECT count(*) FROM raw), 2) AS percent
    FROM raw GROUP BY room_type ORDER BY listings DESC, room_type
""")

| missing_ids | duplicate_keys | excess_duplicate_rows |
| --- | --- | --- |
| 0 | 0 | 0 |

| first_listing_scrape | last_listing_scrape | first_calendar_scrape | last_calendar_scrape |
| --- | --- | --- | --- |
| 2026-06-15 | 2026-06-24 | 2026-06-15 | 2026-06-24 |

| listing_dates_outside_period | calendar_dates_outside_period |
| --- | --- |
| 0 | 0 |

| last_scraped | listings |
| --- | --- |
| 2026-06-15 | 499 |
| 2026-06-16 | 6450 |
| 2026-06-24 | 3420 |

| room_type | listings | percent |
| --- | --- | --- |
| Entire home/apt | 8489 | 81.87 |
| Private room | 1833 | 17.68 |
| Hotel room | 26 | 0.25 |
| Shared room | 21 | 0.2 |

[('Entire home/apt', 8489, 81.87),
 ('Private room', 1833, 17.68),
 ('Hotel room', 26, 0.25),
 ('Shared room', 21, 0.2)]

**Смысл пропусков:** неизвестная цена мешает ценовой медиане, но не подсчёту объявлений;
неизвестный признак календаря мешает основному фильтру доступности. Пропуск числа дней
не равен нулю. Пустые район и тип мешают сравнению групп.

**Наблюдения:** цена отсутствует у 38,50% объявлений — сравнения могут быть смещены;
типы жилья сильно неравномерны по числу объявлений — гостиничные и общие комнаты
образуют малые группы, особенно после деления по районам. Даты сбора не являются
датами бронирований. Точные количества приведены в выводах и отчёте.

## 3. Правила качества
Для каждого правила ниже заданы поле, проверяемые строки, условие, критичность,
действие и порог. Порог оценивает набор; действие применяется к каждой проблемной
строке независимо от превышения порога. Правила предупреждений не используются
для угадывания правильных значений. `source_row_number` идентифицирует строку внутри
снимка; промежуточная таблица материализуется один раз.

In [6]:
con.execute(r"""
    CREATE OR REPLACE TABLE prepared AS
    SELECT *, row_number() OVER () AS source_row_number,
           nullif(trim(id), '') AS listing_id,
           nullif(trim(neighbourhood_cleansed), '') AS district,
           nullif(trim(room_type), '') AS room_type_normalized,
           nullif(trim(price), '') AS price_text,
           try_cast(regexp_replace(trim(price), '[$€,]', '', 'g')
                    AS DECIMAL(14,2)) AS parsed_price,
           coalesce(regexp_full_match(trim(price),
               '[$€]?([0-9]+|[0-9]{1,3}(,[0-9]{3})+)(\.[0-9]{1,2})?'), false)
               AS price_format_ok,
           try_cast(availability_365 AS INTEGER) AS available_days,
           coalesce(regexp_full_match(trim(availability_365), '[0-9]+'), false)
               AS days_format_ok,
           nullif(lower(trim(has_availability)), '') AS calendar_flag,
           count(*) OVER (PARTITION BY nullif(trim(id), '')) AS key_frequency
    FROM raw
""")
rule_results = []


def record_rule(rule_id, criticality, action, threshold):
    assert isinstance(threshold, int) and threshold >= 0
    checked, violations = con.execute(f"""
        SELECT count(*) FILTER (WHERE applicable),
               count(*) FILTER (WHERE applicable AND violation) FROM rule_{rule_id}
    """).fetchone()
    result = dict(
        id=rule_id,
        checked=checked,
        violations=violations,
        share=violations / checked if checked else 0.0,
        threshold=threshold,
        exceeded=violations > threshold,
        criticality=criticality,
        action=action,
    )
    rule_results.append(result)
    show_table(list(result), [list(result.values())])

### `id_integrity` — Ключ объявления
**Поле/связь:** `id`. **Проверяемые строки:** Все строки.
**Условие:** Нормализованный id заполнен и уникален. Все экземпляры повторяющегося ключа попадают в карантин: оснований выбирать один из дублей нет.
**Порог:** 0 нарушивших строк. **Критичность:** `critical`. **Действие:** `quarantine`.

**Связь с задачей:** Идентификация объявления и предотвращение двойного счёта.

In [7]:
threshold_id_integrity = 0
con.execute("""
    CREATE OR REPLACE VIEW rule_id_integrity AS
    SELECT source_row_number, coalesce((
               TRUE
           ), false) AS applicable,
           coalesce((
               listing_id IS NULL
               OR key_frequency > 1
           ), false) AS violation FROM prepared
""")
record_rule("id_integrity", "critical", "quarantine", threshold_id_integrity)

| id | checked | violations | share | threshold | exceeded | criticality | action |
| --- | --- | --- | --- | --- | --- | --- | --- |
| id_integrity | 10369 | 0 | 0.0 | 0 | False | critical | quarantine |

### `district_present` — Полнота района
**Поле/связь:** `neighbourhood_cleansed`. **Проверяемые строки:** Все строки.
**Условие:** Район непустой после trim.
**Порог:** 0 нарушивших строк. **Критичность:** `critical`. **Действие:** `quarantine`.

**Связь с задачей:** Район — основной срез аналитического вопроса.

In [8]:
threshold_district_present = 0
con.execute("""
    CREATE OR REPLACE VIEW rule_district_present AS
    SELECT source_row_number, coalesce((
               TRUE
           ), false) AS applicable,
           coalesce((
               district IS NULL
           ), false) AS violation FROM prepared
""")
record_rule("district_present", "critical", "quarantine", threshold_district_present)

| id | checked | violations | share | threshold | exceeded | criticality | action |
| --- | --- | --- | --- | --- | --- | --- | --- |
| district_present | 10369 | 0 | 0.0 | 0 | False | critical | quarantine |

### `room_type_valid` — Допустимый тип жилья
**Поле/связь:** `room_type`. **Проверяемые строки:** Все строки.
**Условие:** Непустой тип входит в Entire home/apt, Private room, Shared room, Hotel room.
**Порог:** 0 нарушивших строк. **Критичность:** `critical`. **Действие:** `quarantine`.

**Связь с задачей:** Сравниваются одинаковые типы жилья.

In [9]:
threshold_room_type_valid = 0
con.execute("""
    CREATE OR REPLACE VIEW rule_room_type_valid AS
    SELECT source_row_number, coalesce((
               TRUE
           ), false) AS applicable,
           coalesce((
               room_type_normalized IS NULL
               OR room_type_normalized NOT IN (
                   'Entire home/apt', 'Private room', 'Shared room', 'Hotel room')
           ), false) AS violation FROM prepared
""")
record_rule("room_type_valid", "critical", "quarantine", threshold_room_type_valid)

| id | checked | violations | share | threshold | exceeded | criticality | action |
| --- | --- | --- | --- | --- | --- | --- | --- |
| room_type_valid | 10369 | 0 | 0.0 | 0 | False | critical | quarantine |

### `price_present` — Полнота цены
**Поле/связь:** `price`. **Проверяемые строки:** Все строки.
**Условие:** Цена заполнена.
**Порог:** 0 нарушивших строк. **Критичность:** `warning`. **Действие:** `flag`.

**Связь с задачей:** Без цены нельзя включить строку в медиану цены; она сохраняется для числа объявлений и анализа календаря. Нулевой порог фиксирует любую потерю полноты.

In [10]:
threshold_price_present = 0
con.execute("""
    CREATE OR REPLACE VIEW rule_price_present AS
    SELECT source_row_number, coalesce((
               TRUE
           ), false) AS applicable,
           coalesce((
               price_text IS NULL
           ), false) AS violation FROM prepared
""")
record_rule("price_present", "warning", "flag", threshold_price_present)

| id | checked | violations | share | threshold | exceeded | criticality | action |
| --- | --- | --- | --- | --- | --- | --- | --- |
| price_present | 10369 | 3992 | 0.3849937313144951 | 0 | True | warning | flag |

### `price_valid` — Формат и диапазон цены
**Поле/связь:** `price`. **Проверяемые строки:** Только строки с заполненной ценой.
**Условие:** Корректный денежный формат, преобразование в DECIMAL(14,2), значение > 0.
**Порог:** 0 нарушивших строк. **Критичность:** `warning`. **Действие:** `flag`.

**Связь с задачей:** Ошибочная цена исключается из будущей ценовой метрики, но запись сохраняется для остальных характеристик. Символ $ в экспорте Амстердама не меняет валюту EUR.

In [11]:
threshold_price_valid = 0
con.execute("""
    CREATE OR REPLACE VIEW rule_price_valid AS
    SELECT source_row_number, coalesce((
               price_text IS NOT NULL
           ), false) AS applicable,
           coalesce((
               price_text IS NOT NULL
               AND (NOT price_format_ok
               OR parsed_price IS NULL
               OR parsed_price <= 0)
           ), false) AS violation FROM prepared
""")
record_rule("price_valid", "warning", "flag", threshold_price_valid)

| id | checked | violations | share | threshold | exceeded | criticality | action |
| --- | --- | --- | --- | --- | --- | --- | --- |
| price_valid | 6377 | 0 | 0.0 | 0 | False | warning | flag |

### `availability_valid` — Допустимость числа дней
**Поле/связь:** `availability_365`. **Проверяемые строки:** Все строки.
**Условие:** Заполненное целое число от 0 до 365 включительно. Ноль допустим.
**Порог:** 0 нарушивших строк. **Критичность:** `critical`. **Действие:** `quarantine`.

**Связь с задачей:** Число доступных дней — основная аналитическая величина; неизвестное число нельзя заменять нулём.

In [12]:
threshold_availability_valid = 0
con.execute("""
    CREATE OR REPLACE VIEW rule_availability_valid AS
    SELECT source_row_number, coalesce((
               TRUE
           ), false) AS applicable,
           coalesce((
               NOT days_format_ok
               OR available_days IS NULL
               OR available_days NOT BETWEEN 0
               AND 365
           ), false) AS violation FROM prepared
""")
record_rule("availability_valid", "critical", "quarantine", threshold_availability_valid)

| id | checked | violations | share | threshold | exceeded | criticality | action |
| --- | --- | --- | --- | --- | --- | --- | --- |
| availability_valid | 10369 | 0 | 0.0 | 0 | False | critical | quarantine |

### `calendar_flag_valid` — Признак календаря
**Поле/связь:** `has_availability`. **Проверяемые строки:** Все строки.
**Условие:** После нормализации непустое значение t либо f.
**Порог:** 0 нарушивших строк. **Критичность:** `warning`. **Действие:** `flag`.

**Связь с задачей:** Пропуск допускается источником, но ограничивает фильтр основной медианы доступности, поэтому помечается предупреждением.

In [13]:
threshold_calendar_flag_valid = 0
con.execute("""
    CREATE OR REPLACE VIEW rule_calendar_flag_valid AS
    SELECT source_row_number, coalesce((
               TRUE
           ), false) AS applicable,
           coalesce((
               calendar_flag IS NULL
               OR calendar_flag NOT IN ('t', 'f')
           ), false) AS violation FROM prepared
""")
record_rule("calendar_flag_valid", "warning", "flag", threshold_calendar_flag_valid)

| id | checked | violations | share | threshold | exceeded | criticality | action |
| --- | --- | --- | --- | --- | --- | --- | --- |
| calendar_flag_valid | 10369 | 136 | 0.01311601890249783 | 0 | True | warning | flag |

### `calendar_consistency` — Согласованность календаря
**Поле/связь:** `has_availability ↔ availability_365`. **Проверяемые строки:** Строки с f и корректным числом дней.
**Условие:** При f число доступных дней равно 0.
**Порог:** 0 нарушивших строк. **Критичность:** `warning`. **Действие:** `flag`.

**Связь с задачей:** Диагностическая гипотеза о согласованности. Семантика признака может отличаться от числа свободных дней, поэтому строки не исключаются, значения не исправляются. Основной фильтр Brief t не включает эти строки.

In [14]:
threshold_calendar_consistency = 0
con.execute("""
    CREATE OR REPLACE VIEW rule_calendar_consistency AS
    SELECT source_row_number, coalesce((
               calendar_flag = 'f'
               AND days_format_ok
               AND available_days BETWEEN 0
               AND 365
           ), false) AS applicable,
           coalesce((
               calendar_flag = 'f'
               AND days_format_ok
               AND available_days BETWEEN 1
               AND 365
           ), false) AS violation FROM prepared
""")
record_rule("calendar_consistency", "warning", "flag", threshold_calendar_consistency)

| id | checked | violations | share | threshold | exceeded | criticality | action |
| --- | --- | --- | --- | --- | --- | --- | --- |
| calendar_consistency | 152 | 151 | 0.993421052631579 | 0 | True | warning | flag |

In [15]:
show_table(list(rule_results[0]), [list(row.values()) for row in rule_results])
show_sql("""
    SELECT id, price, has_availability, availability_365, neighbourhood_cleansed
    FROM raw WHERE price IS NULL ORDER BY try_cast(id AS BIGINT) LIMIT 2
""")
show_sql("""
    SELECT id, price, has_availability, availability_365, neighbourhood_cleansed FROM raw
    WHERE has_availability = 'f' AND try_cast(availability_365 AS INTEGER) > 0
    ORDER BY try_cast(id AS BIGINT) LIMIT 2
""")

| id | checked | violations | share | threshold | exceeded | criticality | action |
| --- | --- | --- | --- | --- | --- | --- | --- |
| id_integrity | 10369 | 0 | 0.0 | 0 | False | critical | quarantine |
| district_present | 10369 | 0 | 0.0 | 0 | False | critical | quarantine |
| room_type_valid | 10369 | 0 | 0.0 | 0 | False | critical | quarantine |
| price_present | 10369 | 3992 | 0.3849937313144951 | 0 | True | warning | flag |
| price_valid | 6377 | 0 | 0.0 | 0 | False | warning | flag |
| availability_valid | 10369 | 0 | 0.0 | 0 | False | critical | quarantine |
| calendar_flag_valid | 10369 | 136 | 0.01311601890249783 | 0 | True | warning | flag |
| calendar_consistency | 152 | 151 | 0.993421052631579 | 0 | True | warning | flag |

| id | price | has_availability | availability_365 | neighbourhood_cleansed |
| --- | --- | --- | --- | --- |
| 29051 | None | t | 2 | Centrum-Oost |
| 44391 | None | t | 0 | Centrum-Oost |

| id | price | has_availability | availability_365 | neighbourhood_cleansed |
| --- | --- | --- | --- | --- |
| 1034672457625987114 | $143.80 | f | 29 | Centrum-West |
| 1080400468025953047 | $62.81 | f | 215 | Bijlmer-Centrum |

[('1034672457625987114', '$143.80', 'f', '29', 'Centrum-West'),
 ('1080400468025953047', '$62.81', 'f', '215', 'Bijlmer-Centrum')]

## 4. Нормализация, staging и карантин
Карантин имеет приоритет над флагами. Каждая исходная строка попадает ровно в один
набор. `quality_rule_ids` — id критических правил через `;`, причины — в том же порядке.
Исходные поля исключённых строк сохраняются. Даже пустой карантин записывается со схемой.
Денежные значения переводятся в DECIMAL EUR, число дней в INTEGER, календарный признак
в BOOLEAN. Исходные значения преобразованных полей остаются в `*_source`.
Неизвестная/ошибочная цена остаётся NULL с флагом; бизнес-агрегаций нет.

In [16]:
con.execute("""
    CREATE OR REPLACE VIEW checked AS
    SELECT p.*, r0.violation AS id_integrity_violation,
           r1.violation AS district_present_violation,
           r2.violation AS room_type_valid_violation,
           r3.violation AS price_present_violation,
           r4.violation AS price_valid_violation,
           r5.violation AS availability_valid_violation,
           r6.violation AS calendar_flag_valid_violation,
           r7.violation AS calendar_consistency_violation
    FROM prepared p
    LEFT JOIN rule_id_integrity r0 USING (source_row_number)
    LEFT JOIN rule_district_present r1 USING (source_row_number)
    LEFT JOIN rule_room_type_valid r2 USING (source_row_number)
    LEFT JOIN rule_price_present r3 USING (source_row_number)
    LEFT JOIN rule_price_valid r4 USING (source_row_number)
    LEFT JOIN rule_availability_valid r5 USING (source_row_number)
    LEFT JOIN rule_calendar_flag_valid r6 USING (source_row_number)
    LEFT JOIN rule_calendar_consistency r7 USING (source_row_number)
""")
con.execute("""
    CREATE OR REPLACE VIEW classified AS
    SELECT *, concat_ws(';',
        CASE WHEN id_integrity_violation THEN 'id_integrity' END,
        CASE WHEN district_present_violation THEN 'district_present' END,
        CASE WHEN room_type_valid_violation THEN 'room_type_valid' END,
        CASE WHEN availability_valid_violation THEN 'availability_valid' END
    ) AS quality_rule_ids,
    concat_ws('; ',
        CASE WHEN id_integrity_violation THEN 'Пустой или повторяющийся id' END,
        CASE WHEN district_present_violation THEN 'Не указан район' END,
        CASE WHEN room_type_valid_violation THEN 'Неизвестный или пустой тип жилья' END,
        CASE WHEN availability_valid_violation THEN 'Некорректное число дней' END
    ) AS quarantine_reason FROM checked
""")
con.execute("""
    CREATE OR REPLACE TABLE staging AS
    SELECT source_row_number, listing_id AS id, district AS neighbourhood_cleansed,
           room_type_normalized AS room_type,
           CASE WHEN NOT price_present_violation AND NOT price_valid_violation
                THEN parsed_price END AS price_eur,
           available_days AS availability_365,
           CASE WHEN calendar_flag = 't' THEN true
                WHEN calendar_flag = 'f' THEN false END AS has_availability,
           id AS id_source, neighbourhood_cleansed AS neighbourhood_source,
           room_type AS room_type_source, price AS price_source,
           availability_365 AS availability_365_source,
           has_availability AS has_availability_source,
           last_scraped, calendar_last_scraped,
           price_present_violation AS flag_price_missing,
           price_valid_violation AS flag_price_invalid,
           calendar_flag_valid_violation AS flag_calendar_unknown,
           calendar_consistency_violation AS flag_calendar_inconsistent
    FROM classified WHERE quality_rule_ids = '' ORDER BY source_row_number
""")
original_projection = ", ".join('"' + name.replace('"', '""') + '"' for name in raw_columns)
con.execute(f"""
    CREATE OR REPLACE TABLE quarantine AS
    SELECT {original_projection}, source_row_number, quality_rule_ids, quarantine_reason
    FROM classified WHERE quality_rule_ids <> '' ORDER BY source_row_number
""")
show_sql(
    "SELECT (SELECT count(*) FROM raw) AS raw_rows, "
    "(SELECT count(*) FROM staging) AS staging_rows, "
    "(SELECT count(*) FROM quarantine) AS quarantine_rows"
)
show_sql("SELECT id, quality_rule_ids, quarantine_reason FROM quarantine LIMIT 2")

| raw_rows | staging_rows | quarantine_rows |
| --- | --- | --- |
| 10369 | 10369 | 0 |

| id | quality_rule_ids | quarantine_reason |
| --- | --- | --- |


[]

In [17]:
STAGING_PATH = OUTPUT_PREFIX + "staging.parquet"
QUARANTINE_PATH = OUTPUT_PREFIX + "quarantine.parquet"
for table, path in [("staging", STAGING_PATH), ("quarantine", QUARANTINE_PATH)]:
    con.execute(f"COPY {table} TO {sql_literal(path)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    print("Сохранено:", path)

Сохранено: s3://staging/airbnb_amsterdam_listings/source_ingested_on=2026-10-23/staging.parquet
Сохранено: s3://staging/airbnb_amsterdam_listings/source_ingested_on=2026-10-23/quarantine.parquet


## 5. Проверка сохранённых файлов
Оба файла читаются заново напрямую из S3. Проверяются критические правила, флаги,
сохранение строк с учётом кратности и баланс. SHA-256 подтверждает неизменность raw.
В путях Parquet отключено автоматическое добавление разделов пути в схему.

In [18]:
con.execute(
    """CREATE TABLE saved_staging AS
    SELECT * FROM read_parquet(?, hive_partitioning=false)""",
    [STAGING_PATH],
)
con.execute(
    """CREATE TABLE saved_quarantine AS
    SELECT * FROM read_parquet(?, hive_partitioning=false)""",
    [QUARANTINE_PATH],
)
critical_check = show_sql("""
    SELECT count(*) FILTER (WHERE id IS NULL OR trim(id) = '') AS missing_id,
           count(*) - count(DISTINCT id) AS duplicate_id,
           count(*) FILTER (WHERE neighbourhood_cleansed IS NULL
                            OR trim(neighbourhood_cleansed) = '') AS missing_district,
           count(*) FILTER (WHERE room_type IS NULL OR room_type NOT IN
               ('Entire home/apt','Private room','Shared room','Hotel room')) AS bad_room_type,
           count(*) FILTER (WHERE availability_365 IS NULL
                            OR availability_365 NOT BETWEEN 0 AND 365) AS bad_availability
    FROM saved_staging
""")[0]
assert all(value == 0 for value in critical_check), critical_check
flag_check = show_sql(r"""
    SELECT count(*) FILTER (WHERE flag_price_missing IS DISTINCT FROM
                (nullif(trim(price_source), '') IS NULL)) AS price_missing_mismatch,
           count(*) FILTER (WHERE flag_price_invalid IS DISTINCT FROM coalesce(
                nullif(trim(price_source), '') IS NOT NULL AND
                (NOT regexp_full_match(trim(price_source),
                     '[$€]?([0-9]+|[0-9]{1,3}(,[0-9]{3})+)(.[0-9]{1,2})?')
                 OR try_cast(regexp_replace(trim(price_source), '[$€,]', '', 'g')
                             AS DECIMAL(14,2)) IS NULL
                 OR try_cast(regexp_replace(trim(price_source), '[$€,]', '', 'g')
                             AS DECIMAL(14,2)) <= 0), false)) AS price_invalid_mismatch,
           count(*) FILTER (WHERE flag_calendar_unknown IS DISTINCT FROM
                (has_availability IS NULL)) AS calendar_unknown_mismatch,
           count(*) FILTER (WHERE flag_calendar_inconsistent IS DISTINCT FROM
                coalesce(has_availability = false AND availability_365 > 0, false))
                AS calendar_inconsistent_mismatch FROM saved_staging
""")[0]
assert all(value == 0 for value in flag_check), flag_check
show_sql("""
    SELECT count(*) FILTER (WHERE flag_price_missing) AS missing_prices,
           count(*) FILTER (WHERE flag_price_invalid) AS invalid_prices,
           count(*) FILTER (WHERE flag_calendar_unknown) AS unknown_calendars,
           count(*) FILTER (WHERE flag_calendar_inconsistent) AS inconsistent_calendars
    FROM saved_staging
""")
show_sql("""
    SELECT id, price_eur, price_source, flag_price_missing, flag_price_invalid,
           has_availability, availability_365, flag_calendar_unknown,
           flag_calendar_inconsistent FROM saved_staging
    WHERE flag_calendar_inconsistent OR flag_price_missing
    ORDER BY flag_calendar_inconsistent DESC, id LIMIT 2
""")
show_sql("SELECT id, quality_rule_ids, quarantine_reason FROM saved_quarantine LIMIT 2")
staging_count = con.execute("SELECT count(*) FROM saved_staging").fetchone()[0]
quarantine_count = con.execute("SELECT count(*) FROM saved_quarantine").fetchone()[0]
assert raw_count == staging_count + quarantine_count
all_row_ids = con.execute("""
    SELECT count(*), count(DISTINCT source_row_number),
           min(source_row_number), max(source_row_number)
    FROM (SELECT source_row_number FROM saved_staging UNION ALL
          SELECT source_row_number FROM saved_quarantine)
""").fetchone()
assert all_row_ids[:2] == (raw_count, raw_count)
if raw_count:
    assert all_row_ids[2:] == (1, raw_count)
assert (
    con.execute("""
    SELECT count(*) FROM saved_quarantine
    WHERE quality_rule_ids IS NULL OR quality_rule_ids = ''
       OR quarantine_reason IS NULL OR quarantine_reason = ''
""").fetchone()[0]
    == 0
)
for table in ("staging", "quarantine"):
    assert (
        con.execute(f"""
        SELECT count(*) FROM (
            (SELECT * FROM {table} EXCEPT ALL SELECT * FROM saved_{table}) UNION ALL
            (SELECT * FROM saved_{table} EXCEPT ALL SELECT * FROM {table})
        )
    """).fetchone()[0]
        == 0
    )
raw_after = raw_fingerprint()
assert raw_before == raw_after, "Исходный объект изменился"
run_result = {
    "execution_environment": EXECUTION_ENVIRONMENT,
    "input": RAW_PATH,
    "period": ANALYTICAL_PERIOD,
    "output_prefix": OUTPUT_PREFIX,
    "raw": raw_count,
    "staging": staging_count,
    "quarantine": quarantine_count,
    "raw_fingerprint": raw_after,
    "critical_violations": list(critical_check),
    "flag_mismatches": list(flag_check),
    "rules": rule_results,
}
print("RUN_RESULT=" + json.dumps(run_result, ensure_ascii=False, sort_keys=True))
show_table(
    ["raw", "staging", "карантин", "Баланс", "raw неизменен"],
    [(raw_count, staging_count, quarantine_count, True, True)],
)
con.close()

| missing_id | duplicate_id | missing_district | bad_room_type | bad_availability |
| --- | --- | --- | --- | --- |
| 0 | 0 | 0 | 0 | 0 |

| price_missing_mismatch | price_invalid_mismatch | calendar_unknown_mismatch | calendar_inconsistent_mismatch |
| --- | --- | --- | --- |
| 0 | 0 | 0 | 0 |

| missing_prices | invalid_prices | unknown_calendars | inconsistent_calendars |
| --- | --- | --- | --- |
| 3992 | 0 | 136 | 151 |

| id | price_eur | price_source | flag_price_missing | flag_price_invalid | has_availability | availability_365 | flag_calendar_unknown | flag_calendar_inconsistent |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1034672457625987114 | 143.80 | $143.80 | False | False | False | 29 | False | True |
| 1080400468025953047 | 62.81 | $62.81 | False | False | False | 215 | False | True |

| id | quality_rule_ids | quarantine_reason |
| --- | --- | --- |


RUN_RESULT={"critical_violations": [0, 0, 0, 0, 0], "execution_environment": "project S3", "flag_mismatches": [0, 0, 0, 0], "input": "s3://raw/airbnb_amsterdam_listings/ingested_on=2026-10-23/listings.csv.gz", "output_prefix": "s3://staging/airbnb_amsterdam_listings/source_ingested_on=2026-10-23/", "period": ["2026-06-15", "2026-06-24"], "quarantine": 0, "raw": 10369, "raw_fingerprint": {"bytes": 5759896, "sha256": "e67b01d8b46e04f8ea85cd401dc6deeadef3b482fccf39e0388020d63c394062"}, "rules": [{"action": "quarantine", "checked": 10369, "criticality": "critical", "exceeded": false, "id": "id_integrity", "share": 0.0, "threshold": 0, "violations": 0}, {"action": "quarantine", "checked": 10369, "criticality": "critical", "exceeded": false, "id": "district_present", "share": 0.0, "threshold": 0, "violations": 0}, {"action": "quarantine", "checked": 10369, "criticality": "critical", "exceeded": false, "id": "room_type_valid", "share": 0.0, "threshold": 0, "violations": 0}, {"action": "flag",

| raw | staging | карантин | Баланс | raw неизменен |
| --- | --- | --- | --- | --- |
| 10369 | 10369 | 0 | True | True |

## 6. Выводы и повторяемость
Флаги не означают автоматического удаления объявления. Ценовые выводы относятся
к известным корректным ценам, основной анализ календаря — к `has_availability=true`.
Недоступность не доказывает бронирование, один снимок не описывает динамику рынка.

Выполните **Restart Kernel → Run All** повторно. Сравните `RUN_RESULT` двух запусков
и запишите количества в отчёт. Скрипт `scripts/verify_lab03.py` автоматизирует два
запуска в новых Jupyter-ядрах. Аналитический SQL целиком остаётся в ноутбуке.